# `StructureSurface` example: multi-protein complex (NPM1 pentamer)

The NPM1 pentamer is supplied as an AlphaFold3 **mmCIF** file
(`fold_npm1_pentamer_model_0.cif`), demonstrating that `StructureSurface` handles
both PDB and mmCIF/PDBx via mdtraj. The N-terminal oligomerization domain forms a
5-chain pentamer with disordered tails, which is ideal for exercising:

1. Per-chain folded/IDR decomposition.
2. Surface classification across all chains.
3. A *cross-chain* contiguous-surface net (the pentamer interface).
4. FINCHES surface scoring against an IDR + surface-vs-surface contacts.
5. cis vs trans reach (same chain vs neighbouring chains).

In [1]:
from finches.frontend.mpipi_frontend import Mpipi_frontend
from finches.utils.structure_surface import StructureSurface
import numpy as np

mf = Mpipi_frontend()
ss = StructureSurface("fold_npm1_pentamer_model_0.cif", mf)

## 1. Per-chain decomposition

In [2]:
chains = sorted({r.chain_index for r in ss.residues})
print(f"{'chain':>5} {'residues':>9} {'folded':>7} {'idr':>5} {'surface':>8}")
for c in chains:
    recs   = [r for r in ss.residues if r.chain_index == c]
    folded = sum(1 for r in recs if r.domain == 'folded' and r.modeled)
    idr    = sum(1 for r in recs if r.domain == 'idr')
    surf   = sum(1 for r in recs if r.surface)
    print(f"{c:>5} {len(recs):>9} {folded:>7} {idr:>5} {surf:>8}")
print(f"\nIDR segments across all chains: {len(ss.idr_segments)}")

chain  residues  folded   idr  surface
    0       294     144   150      101
    1       294     144   150      101
    2       294     144   150      101
    3       294     144   150      101
    4       294     144   150      101

IDR segments across all chains: 10


## 2 + 3. Surface + cross-chain net
The pentamer oligomerization interface shows up as **cross-chain** edges in the surface net (residues on different chains that share a contiguous solvent-accessible surface).

In [3]:
g = ss.surface_graph
cross = [(a, b) for a, b in g.edges if a[0] != b[0]]
print(f"surface residues : {len(ss.surface_residues)}")
print(f"net edges        : {g.number_of_edges()}")
print(f"cross-chain edges: {len(cross)}  (oligomerization interface contacts)")

surface residues : 505
net edges        : 759
cross-chain edges: 114  (oligomerization interface contacts)


## 4. FINCHES scoring against an IDR
NPM1's surface is highly acidic, so a basic (Arg-rich) IDR should be broadly attractive.

In [4]:
basic_idr = "RGRGRGRGRGRGRGRGRGRG"
scores = ss.surface_vs_idr(basic_idr)
vals   = np.array([v['score'] for v in scores.values()])
print(f"mean score over surface : {vals.mean():+.3f}  (negative = attractive)")
print(f"fraction attractive     : {(vals < 0).mean():.0%}")

mean score over surface : -0.178  (negative = attractive)
fraction attractive     : 82%


### Strongest surface-vs-surface contacts
Each surface residue carries its own surface-patch context. The strongest contacts are interfacial salt bridges, including cross-chain pairs.

In [5]:
svs = ss.surface_vs_surface()
strongest = sorted(svs.items(), key=lambda kv: kv[1])[:6]
for (ka, kb), val in strongest:
    a = ss.get_residue(*ka); b = ss.get_residue(*kb)
    tag = "cross-chain" if ka[0] != kb[0] else "same-chain"
    print(f"{a.one_letter}{ka[1]}(ch{ka[0]}) <-> "
          f"{b.one_letter}{kb[1]}(ch{kb[0]})  {val:+.3f}  [{tag}]")

K24(ch0) <-> D26(ch0)  -2.673  [same-chain]
K24(ch0) <-> D36(ch0)  -2.673  [same-chain]
K24(ch0) <-> D55(ch0)  -2.673  [same-chain]
K24(ch0) <-> D26(ch1)  -2.673  [cross-chain]
K24(ch0) <-> D36(ch1)  -2.673  [cross-chain]
K24(ch0) <-> D55(ch1)  -2.673  [cross-chain]


## 5. cis vs trans reach
An IDR anchored on one chain can reach surface residues on its own chain (cis) and on neighbouring chains (trans).

In [6]:
seg = max((s for s in ss.idr_segments if s['n_anchor'] or s['c_anchor']),
          key=lambda s: len(s['residues']))
anchor  = (seg['c_anchor'] or seg['n_anchor']).key
idr_len = max(len(seg['residues']), 40)

reachable = ss.reachable_surface_residues(anchor, idr_len)
cis   = sum(1 for k in reachable if k[0] == anchor[0])
trans = len(reachable) - cis
print(f"IDR length {idr_len} anchored at {anchor}")
print(f"  reachable surface residues : {len(reachable)} (cis={cis}, trans={trans})")

IDR length 137 anchored at (0, 253)
  reachable surface residues : 504 (cis=101, trans=403)
